# SENTINEL GRPO Training (Colab T4)

This notebook trains a small GRPO LoRA, records a deterministic replay table, and generates the full SENTINEL demo chart bundle for the Hugging Face Space.

In [ ]:
!nvidia-smi
!git clone https://github.com/ADITYAGABA1322/sentinel-env
%cd sentinel-env
!pip install -q -r requirements.txt
!pip install -q -r requirements-train.txt

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
!python -m pytest -q
!python training/evaluate.py --episodes 30 --task all --out outputs/eval_pre.json --no-plot

In [ ]:
!python training/train.py \
  --episodes 200 --task all --seed 0 \
  --model unsloth/Qwen2.5-1.5B-Instruct \
  --epochs 1 --batch-size 2 --learning-rate 5e-6 \
  --lora-rank 16 --max-seq-length 1024 \
  --output-dir training/sentinel_qwen15_grpo

In [ ]:
from training.replay import record_trained_actions

record_trained_actions(
    adapter_path="training/sentinel_qwen15_grpo",
    base_model="unsloth/Qwen2.5-1.5B-Instruct",
    tasks=["task1", "task2", "task3"],
    seeds=range(30),
    out_path="outputs/trained_policy_replay.jsonl",
)

In [ ]:
!python training/evaluate.py --episodes 30 --task all \
  --policies random,heuristic,oracle_lite,trained \
  --replay outputs/trained_policy_replay.jsonl \
  --out outputs/eval_post.json --no-plot

In [ ]:
!python -m training.plots \
  --pre outputs/eval_pre.json \
  --post outputs/eval_post.json \
  --trainer-state training/sentinel_qwen15_grpo/trainer_state.json \
  --reward-report-task3 outputs/reward_report_task3_seed42.json \
  --cluster-health outputs/cluster_health_history.json \
  --out-dir outputs/charts

In [ ]:
from IPython.display import Image, display
for name in [
    "baseline_grouped_bars.png",
    "grpo_reward_curve.png",
    "trust_evolution.png",
    "detection_vs_poisoning.png",
    "cluster_health_timeline.png",
    "task_radar.png",
    "ablation.png",
    "baseline_delta_lines.png",
    "cluster_health_policy_lines.png",
    "trust_gap_over_time.png",
    "reward_component_stacked_area.png",
    "failure_fishbone_map.png",
]:
    print(name)
    display(Image(f"outputs/charts/{name}"))

In [ ]:
from huggingface_hub import HfApi
api = HfApi()
api.create_repo("XcodeAddy/sentinel-grpo-qwen15", exist_ok=True)
api.upload_folder(
    folder_path="training/sentinel_qwen15_grpo",
    repo_id="XcodeAddy/sentinel-grpo-qwen15",
)
print("Uploaded LoRA adapter. Commit outputs/charts/*.png and outputs/trained_policy_replay.jsonl back to the repo.")